# Lesson 15: Edge AI Agents & SLM Function Calling — Deterministic JSON Schemas, Tool Loops & Guardrails

In this companion notebook, you will:
1. Define strict JSON tool schemas for an offline edge diagnostic agent (`read_sensor`, `set_fan_speed`).
2. Execute the **6-Step SLM Function Calling Loop** with parameter validation, rate limiting, and audit logging.
3. Test tool-calling prompts using an open-source local SLM (`Qwen/Qwen2.5-0.5B-Instruct`) on Google Colab.

In [ ]:
!pip install -q -U transformers torch pydantic

## 1. Define Typed Edge Tools & Safety Guardrails
Using Pydantic ensures malformed SLM outputs never trigger unsafe hardware actions.

In [ ]:
import json
from pydantic import BaseModel, Field, ValidationError

class FanSpeedArgs(BaseModel):
    rpm: int = Field(ge=500, le=5000, description="Safe fan speed in RPM")

def set_fan_speed(rpm: int) -> dict:
    return {"status": "ok", "fan_rpm": rpm}

def dispatch_tool(name: str, raw_args: str) -> dict:
    if name != "set_fan_speed":
        return {"error": f"Unknown tool: {name}"}
    try:
        args = FanSpeedArgs(**json.loads(raw_args))
        return set_fan_speed(args.rpm)
    except (json.JSONDecodeError, ValidationError) as exc:
        return {"error": f"Validation blocked unsafe call: {exc}"}

print("Valid call:  ", dispatch_tool("set_fan_speed", '{"rpm": 3200}'))
print("Unsafe call: ", dispatch_tool("set_fan_speed", '{"rpm": 99999}'))